# Spatially anisotropic firing: direction-resolved synchrony over distance

Self-contained analysis of how the spatial **anisotropy of coordinated firing**
develops over DIV in the `stim_removal_null` dataset (the recordings Figure 4
compiles). It keeps the *orientation* of every electrode pair — which the scalar
synchrony-vs-distance pipeline discards — so anisotropy becomes measurable.

The heavy lifting lives in the tested module `ephax.metrics.directional_synchrony`;
this notebook only loads data, drives the sweep, and draws panels.

**Panels**
1. Anisotropy magnitude vs DIV, aggregated across wells (de-confounded cos2θ
   amplitude, "option A"), under the conservative interval-jitter null.
2. Axis-aligned orientation curves per DIV (each well rotated to its own axis,
   then pooled) — shows the orientation-dependent radial reach emerging over DIV.
3. Orientation-resolved conditional firing probability and excess synchrony vs
   distance for one recording (cyclic colormap; raw, not demeaned, so the radial
   structure is preserved).

> Each well has its **own** anisotropy axis, so we aggregate the axis-invariant
> *magnitude* across wells and only pool the *curves* after aligning each well to
> its own axis — pooling raw orientations would cancel the signal.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

repo_root = Path.cwd()
if not (repo_root / "ephax").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from ephax.metrics.directional_synchrony import (
    DirectionalSyncConfig,
    compute_pair_synchrony,
    cos2_anisotropy_index,
    orientation_shuffle_pvalue,
    align_to_axis,
    orientation_distance_profile,
    aggregate_amplitude,
)

## Configuration

In [ ]:
DATA_ROOT = repo_root / "ephax" / "data" / "stimRemovalNull"
FILENAME_TEMPLATE = "DIV{div}_240703_data_well{well}_exp_data.npz"

DIVS = [12, 14, 21, 22, 23]
WELLS = [0, 1, 2, 3, 4, 5]
WINDOW_S = (0.0, 300.0)          # START_SEC, END_SEC (matches the Figure-4 pipeline)
TOP_N_ELECTRODES = 1000          # TOP_STOP for stim_removal_null

# Anisotropy index is fit over a coverage-complete distance range (full orientation
# coverage needs distances within the array's short dimension). The radial curves
# still run to MAX_DISTANCE_UM so the ~2540 um model wavelength stays visible.
INDEX_DISTANCE_UM = (150.0, 2000.0)
RING_UM = 200.0

# High-activity interval detection (matches activity_distance_spike.ipynb defaults).
NET_BIN_MS = 10.0
HIGH_ACTIVITY_MAD_SCALE = 3.0
HIGH_ACTIVITY_MIN_DURATION_MS = 30.0

# Synchrony / null settings. interval_jitter is the conservative confirmatory null;
# switch to "rate_expectation" for a fast (~1 s/recording) preview.
SYNC = DirectionalSyncConfig(
    matrix_bin_ms=1.0,
    lag_window_ms=(-5.0, 5.0),
    null_method="interval_jitter",
    jitter_ms=25.0,
    n_surrogates=50,
    min_trigger_spikes=5,
    min_distance_um=50.0,
    max_distance_um=3500.0,
    n_orientation_bins=12,
    index_distance_um=INDEX_DISTANCE_UM,
    index_ring_um=RING_UM,
    random_seed=0,
)

# Fast preview: subset wells/DIVs and use the closed-form null. Set False for the
# full jitter result (~15-20 min for 30 recordings).
SMOKE = True
if SMOKE:
    DIVS = [12, 21]
    WELLS = [0, 1, 2]
    SYNC = DirectionalSyncConfig(**{**SYNC.__dict__, "null_method": "rate_expectation"})

OUTPUT_DIR = repo_root / "outputs" / "directional_synchrony"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CMAP = plt.get_cmap("twilight")   # cyclic: maps axial orientation so 0 deg == 180 deg

## Data loading and preprocessing helpers

In [ ]:
def load_recording(path):
    """Load spike times, per-spike electrode, and electrode->(x,y) from an NPZ."""
    d = np.load(path, allow_pickle=True)
    sf = float(np.asarray(d["samp_rate"]).reshape(-1)[0])
    cm = np.asarray(d["channelmap"], dtype=float)   # columns: channel, _, electrode, x, y
    ch_to_elec = {int(c): int(e) for c, e in zip(cm[:, 0], cm[:, 2])
                  if np.isfinite(c) and np.isfinite(e)}
    coords = {int(e): (x, y) for e, x, y in zip(cm[:, 2], cm[:, 3], cm[:, 4])
              if np.isfinite(e) and np.isfinite(x) and np.isfinite(y)}
    sd = d["spike_data"]
    times = np.asarray(sd["frameno"], dtype=float) / sf
    chans = np.asarray(sd["channel"], dtype=int)
    elec = np.fromiter((ch_to_elec.get(int(c), -1) for c in chans), dtype=int, count=chans.size)
    keep = elec >= 0
    times, elec = times[keep], elec[keep]
    times = times - times.min()                      # rebase clock to spike onset
    return times, elec, coords


def detect_high_activity_intervals(times, t0, t1):
    """Population-rate MAD threshold -> merged high-activity (start, end) intervals."""
    bin_s = NET_BIN_MS / 1000.0
    n_bins = int(np.ceil((t1 - t0) / bin_s))
    counts = np.zeros(n_bins)
    idx = np.floor((times - t0) / bin_s).astype(int)
    ok = (idx >= 0) & (idx < n_bins)
    np.add.at(counts, idx[ok], 1.0)
    med = np.median(counts)
    mad = np.median(np.abs(counts - med)) or 1.0
    thr = med + HIGH_ACTIVITY_MAD_SCALE * 1.4826 * mad
    active = counts > thr
    min_bins = max(1, int(round(HIGH_ACTIVITY_MIN_DURATION_MS / NET_BIN_MS)))
    intervals, i = [], 0
    while i < n_bins:
        if active[i]:
            j = i
            while j < n_bins and active[j]:
                j += 1
            if (j - i) >= min_bins:
                intervals.append((t0 + i * bin_s, t0 + j * bin_s))
            i = j
        else:
            i += 1
    return intervals


def select_top_electrodes(times, elec, coords, n):
    """Top-n electrodes by spike count that have coordinates (activity scan)."""
    uniq = np.unique(elec)
    uniq = uniq[np.array([int(e) in coords for e in uniq])]
    counts = np.array([np.count_nonzero(elec == e) for e in uniq])
    return uniq[np.argsort(counts)[::-1][:n]]


def recording_path(well, div):
    return DATA_ROOT / f"well{well}" / FILENAME_TEMPLATE.format(div=div, well=well)

## Per-recording directional synchrony sweep

For each (DIV, well): select the top electrodes, detect high-activity intervals,
compute per-pair excess synchrony with orientation retained, then the de-confounded
cos2θ anisotropy amplitude/axis and the axis-aligned orientation×distance profiles.


In [ ]:
records = []          # one row per (div, well): amplitude, axis, n_pairs
ex_grids = {d: [] for d in DIVS}   # axis-aligned excess profiles
po_grids = {d: [] for d in DIVS}   # axis-aligned p_obs profiles
dist_edges = np.arange(SYNC.min_distance_um, SYNC.max_distance_um + RING_UM, RING_UM)
dist_centers = 0.5 * (dist_edges[:-1] + dist_edges[1:])
N_CURVE_BINS = 6

for div in DIVS:
    for well in WELLS:
        path = recording_path(well, div)
        times, elec, coords = load_recording(path)
        win = (times >= WINDOW_S[0]) & (times <= WINDOW_S[1])
        times, elec = times[win], elec[win]
        selected = select_top_electrodes(times, elec, coords, TOP_N_ELECTRODES)
        intervals = detect_high_activity_intervals(times, float(times.min()), float(times.max()))

        ps = compute_pair_synchrony(times, elec, coords, selected, intervals, SYNC,
                                    np.random.default_rng(SYNC.random_seed))
        idx = cos2_anisotropy_index(ps.excess, ps.distance_um, ps.angle_rad,
                                    distance_range=INDEX_DISTANCE_UM, ring_um=RING_UM)
        records.append({"div": div, "well": well, "amplitude": idx["amplitude"],
                        "axis_deg": idx["axis_deg"], "n_pairs": idx["n_pairs"],
                        "active_s": sum(b - a for a, b in intervals)})
        if np.isfinite(idx["axis_rad"]):
            rel = align_to_axis(ps.angle_rad, idx["axis_rad"])   # orientation vs this well's axis
            ex_grids[div].append(orientation_distance_profile(
                ps.excess, ps.distance_um, rel, n_orientation_bins=N_CURVE_BINS,
                distance_edges=dist_edges, min_pairs_per_cell=20)["grid"])
            po_grids[div].append(orientation_distance_profile(
                ps.p_obs, ps.distance_um, rel, n_orientation_bins=N_CURVE_BINS,
                distance_edges=dist_edges, min_pairs_per_cell=20)["grid"])
        print(f"DIV{div} well{well}: amp={idx['amplitude']:.4f} axis={idx['axis_deg']:.0f} deg "
              f"n_pairs={idx['n_pairs']}", flush=True)

index_df = pd.DataFrame(records)
index_df.to_csv(OUTPUT_DIR / "anisotropy_index_by_recording.csv", index=False)
index_df

## Panel 1 — anisotropy magnitude vs DIV, aggregated across wells

In [ ]:
agg = {div: aggregate_amplitude(index_df.loc[index_df["div"] == div, "amplitude"]) for div in DIVS}
means = [agg[d]["mean"] for d in DIVS]
sems = [agg[d]["sem"] for d in DIVS]

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.errorbar(DIVS, means, yerr=sems, fmt="o-", lw=1.8, ms=6, color="#7b61ff", capsize=3,
            label="mean ± SEM across wells")
for div in DIVS:
    a = index_df.loc[index_df["div"] == div, "amplitude"].to_numpy(float)
    ax.plot([div] * a.size, a, "o", ms=3, color="0.6", alpha=0.6, zorder=0)
ax.set_xlabel("DIV")
ax.set_ylabel("synchrony anisotropy amplitude (cos2θ, option A)")
ax.set_title(f"Anisotropy magnitude vs DIV ({SYNC.null_method} null)\n"
             f"de-confounded {INDEX_DISTANCE_UM[0]:.0f}-{INDEX_DISTANCE_UM[1]:.0f} µm")
ax.set_xticks(DIVS)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "anisotropy_amplitude_vs_div.png", dpi=130)
plt.show()

## Panel 2 — axis-aligned orientation curves per DIV

Each well is rotated to its own anisotropy axis, then profiles are averaged across
wells. Relative orientation 0°/180° is **along** the well axis; 90° is perpendicular.
Excess synchrony on a symlog axis (keeps the long-range below-chance troughs).


In [ ]:
fig, axes = plt.subplots(1, len(DIVS), figsize=(3.6 * len(DIVS), 4.0), sharey=True)
axes = np.atleast_1d(axes)
ang_edges = np.linspace(0, np.pi, N_CURVE_BINS + 1)
for ax, div in zip(axes, DIVS):
    if ex_grids[div]:
        grid = np.nanmean(np.array(ex_grids[div]), axis=0)   # average over wells
        for a in range(N_CURVE_BINS):
            ax.plot(dist_centers, grid[a], "-o", ms=2.5, lw=1.1, color=CMAP((a + 0.5) / N_CURVE_BINS))
    ax.set_yscale("symlog", linthresh=0.01)
    ax.axhline(0, color="0.6", lw=0.6)
    ax.set_title(f"DIV {div}", fontsize=10)
    ax.set_xlabel("distance (µm)")
axes[0].set_ylabel("excess synchrony (axis-aligned)")
sm = ScalarMappable(norm=Normalize(0, 180), cmap=CMAP)
cb = fig.colorbar(sm, ax=axes, shrink=0.7, pad=0.01)
cb.set_label("orientation relative to well axis (deg)")
cb.set_ticks([0, 90, 180])
fig.suptitle(f"Axis-aligned, well-pooled excess synchrony vs distance ({SYNC.null_method} null)")
fig.savefig(OUTPUT_DIR / "aligned_orientation_curves_by_div.png", dpi=120, bbox_inches="tight")
plt.show()

## Panel 3 — orientation-resolved synchrony for one recording

Both quantities that fall out of the same computation: the raw conditional firing
probability p(j|i) (log-y) and the derived excess synchrony (symlog). Orientation
is the pair's own axial angle (not aligned), colored with the cyclic colormap.
The radial profile is kept raw (not demeaned) so the distance structure is visible.


In [ ]:
# pick the strongest recording for illustration
best = index_df.loc[index_df["amplitude"].idxmax()]
div_x, well_x = int(best["div"]), int(best["well"])
times, elec, coords = load_recording(recording_path(well_x, div_x))
win = (times >= WINDOW_S[0]) & (times <= WINDOW_S[1]); times, elec = times[win], elec[win]
selected = select_top_electrodes(times, elec, coords, TOP_N_ELECTRODES)
intervals = detect_high_activity_intervals(times, float(times.min()), float(times.max()))
ps = compute_pair_synchrony(times, elec, coords, selected, intervals, SYNC,
                            np.random.default_rng(SYNC.random_seed))

prof_po = orientation_distance_profile(ps.p_obs, ps.distance_um, ps.angle_rad,
                                       n_orientation_bins=N_CURVE_BINS, distance_edges=dist_edges)
prof_ex = orientation_distance_profile(ps.excess, ps.distance_um, ps.angle_rad,
                                       n_orientation_bins=N_CURVE_BINS, distance_edges=dist_edges)

fig, axs = plt.subplots(1, 2, figsize=(12, 4.6))
for a in range(N_CURVE_BINS):
    color = CMAP((a + 0.5) / N_CURVE_BINS)
    axs[0].plot(dist_centers, prof_po["grid"][a], "-o", ms=3, lw=1.2, color=color)
    axs[1].plot(dist_centers, prof_ex["grid"][a], "-o", ms=3, lw=1.2, color=color)
axs[0].set_yscale("log"); axs[0].set_title("p(j fires | i fires)  (log-y)")
axs[0].set_ylabel("conditional firing prob")
axs[1].set_yscale("symlog", linthresh=0.01); axs[1].axhline(0, color="0.6", lw=0.6)
axs[1].set_title("excess synchrony  (symlog)"); axs[1].set_ylabel("excess synchrony")
for ax in axs:
    ax.set_xlabel("distance (µm)"); ax.set_xlim(0, SYNC.max_distance_um)
sm = ScalarMappable(norm=Normalize(0, 180), cmap=CMAP)
cb = fig.colorbar(sm, ax=axs, shrink=0.8, pad=0.02)
cb.set_label("pair orientation (deg, axial)"); cb.set_ticks([0, 90, 180])
fig.suptitle(f"Orientation-resolved synchrony — DIV{div_x} well{well_x} ({SYNC.null_method} null)")
fig.savefig(OUTPUT_DIR / "orientation_resolved_synchrony_example.png", dpi=120, bbox_inches="tight")
plt.show()

## Notes

* **Null.** `interval_jitter` (the default here) is the conservative surrogate null;
  it agrees with the fast `rate_expectation` null on the anisotropy magnitude while
  controlling slow rate covariation. Set `SMOKE = True` for a quick `rate_expectation`
  preview on a subset.
* **Why aggregate magnitude, not raw curves.** Each well has its own anisotropy axis,
  so the across-well aggregate uses the axis-invariant cos2θ *amplitude*; the curves are
  pooled only after aligning each well to its own axis.
* **Distance ranges.** The cos2θ index is fit over a coverage-complete range
  (`INDEX_DISTANCE_UM`); beyond the array's short dimension only near-axis pairs exist,
  so the angular fit there is geometrically ill-posed. The radial curves still extend to
  `max_distance_um` to show the long-range structure.
* **Significance.** `orientation_shuffle_pvalue` gives a per-recording one-sided p-value
  by permuting orientation labels and refitting.
